### Countries Population UseCase
[Source](https://github.com/azurede007/data-repo/blob/main/countries_population.csv)
- COUNTRY_ID – Unique identifier for each country.
- NAME – Official name of the country.
- NATIONALITY – Name of the citizens of the country.
- COUNTRY_CODE – Numeric or internal code representing the country.
- ISO_ALPHA2 – Two-letter ISO standard code for the country.
- CAPITAL – Capital city of the country.
- POPULATION – Total number of people living in the country.
- AREA_KM2 – Total land area of the country in square kilometers.
- REGION_ID – Identifier for the region the country belongs to.
- SUB_REGION_ID – Identifier for the sub-region within the region.

### UseCases

- Top Capitals – Identify capitals with the largest populations.
- Country Count per Region – Count how many countries belong to each region.
- Population Ranking – List countries from highest to lowest population.
- Region Population Summary – Sum population per region and sub-region.

In [0]:
from pyspark.sql.functions import col,max,count,row_number,sum,round
from pyspark.sql.window import Window
from pyspark.sql.types import *
count_sch= StructType([StructField('COUNTRY_ID', IntegerType(), True), StructField('NAME', StringType(), True), StructField('NATIONALITY', StringType(), True), StructField('COUNTRY_CODE', StringType(), True), StructField('ISO_ALPHA2', StringType(), True), StructField('CAPITAL', StringType(), True), StructField('POPULATION', FloatType(), True), StructField('AREA_KM2', StringType(), True), StructField('REGION_ID', IntegerType(), True), StructField('SUB_REGION_ID', IntegerType(), True)])

df=spark.read.format("csv").option("header",True).schema(count_sch).load("/Volumes/databricks_practice/inputdb/country_population/countries_population.csv")
#print(df.schema)
df.show(5)
#Top Capitals – Identify capitals with the largest populations.
top_caps_df=df.agg(max("POPULATION"))
#top_caps_df.display()
max_pop = top_caps_df.collect()[0][0]  
#print(max_pop)
#df.filter(col("POPULATION")==max_pop).show()
#df.orderBy(df.POPULATION.desc()).limit(1).show()
#Country Count per Region – Count how many countries belong to each region.
count_count_df=df.groupBy("REGION_ID").agg(count("COUNTRY_ID").alias("country_count"))
#count_count_df.show()
#Population Ranking – List countries from highest to lowest population.
#df.withColumn("POPULATION_RANK_ID",row_number().over(Window.orderBy(col("POPULATION").desc()))).show(5)
win_spec=Window.orderBy(col("POPULATION").desc())
pop_rank_df=df.withColumn("POPULATION_RANK_ID",row_number().over(win_spec))
pop_rank_df.orderBy(col("POPULATION").desc()).show(5)
#Region Population Summary – Sum population per region and sub-region.
df.groupBy("REGION_ID","SUB_REGION_ID").agg(round(sum(col("POPULATION")),2).alias("REG_POP")).show()
